<a href="https://colab.research.google.com/github/Syeda-Rb-47/Building-GlueScore--An-Interpretable-Glue-Likeness-Scoring-Pipeline/blob/main/3_Feature_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORTS

In [ ]:
# Installing RDKit and importing required libraries.
!pip install -q rdkit

import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Defining dataset path and thresholds for descriptor filtering and IMID bias detection.
DRIVE_DIR     = "/content/drive/MyDrive/Thesis Datasets"
D1_THRESHOLD  = 0.5  # max |Cohen's d| for non-IMID vs BG (should be similar)
D2_THRESHOLD  = 0.5  # min |Cohen's d| for IMID vs BG (should be the outlier)

In [ ]:
# Loading the dataset
df = pd.read_csv(DRIVE_DIR + "/merged_cleaned_dataset.csv")
print("Dataset loaded:", df.shape)
print(df["Label"].value_counts())

Dataset loaded: (3742, 5)
Label
0    2118
1    1624
Name: count, dtype: int64


CALCULATING DESCRIPTOR VALUES

In [ ]:
# Getting full descriptor list and functions directly from RDKit.
desc_names = [d[0] for d in Descriptors.descList]
desc_funcs = [d[1] for d in Descriptors.descList]

# Defining and implementing descriptor calculation function.
def calculate_descriptors(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return [None] * len(desc_names)
        results = []
        for func in desc_funcs:
            try:
                results.append(func(mol))
            except:
                results.append(None)
        return results
    except:
        return [None] * len(desc_names)

print("Calculating", len(desc_names), "descriptors for", len(df), "molecules...")
desc_values         = df["Canonical_SMILES"].apply(calculate_descriptors).apply(pd.Series)
desc_values.columns = desc_names
desc_values         = desc_values.replace([np.inf, -np.inf], np.nan)

# Combining descriptors with metadata columns.
desc_matrix = pd.concat([df[["Master_ID", "Source_ID", "Label", "StdInChIKey", "Canonical_SMILES"]], desc_values], axis=1)

# Dropping rows where any descriptor failed to calculate.
before      = len(desc_matrix)
desc_matrix = desc_matrix.dropna(subset=desc_names).reset_index(drop=True)
print("Rows dropped (failed descriptor calculation):", before - len(desc_matrix))
print("Descriptor matrix shape:", desc_matrix.shape)

Calculating 217 descriptors for 3742 molecules...
Rows dropped (failed descriptor calculation): 4
Descriptor matrix shape: (3738, 222)


In [ ]:
# Checking which rows were dropped and why.
failed_mask = desc_values.isnull().any(axis=1)
failed_rows = df[failed_mask]
print("Dropped molecules:")
print(failed_rows[["Master_ID", "Source_ID", "Label", "Canonical_SMILES"]])

print("\nDescriptors that caused failures:")
print(desc_values[failed_mask].isnull().sum()[desc_values[failed_mask].isnull().sum() > 0])

Dropped molecules:
      Master_ID      Source_ID  Label  \
1331       1332           1336      1   
2082       2083  CHEMBL1200928      0   
2135       2136  CHEMBL1201015      0   
2388       2389  CHEMBL1330792      0   

                                       Canonical_SMILES  
1331  COc1cc2cc(C(=O)NCc3ccc4c(c3)CN(C3CCC(=O)NC3=O)...  
2082                                             [Zn+2]  
2135                                             [Al+3]  
2388                     CC(=O)Nc1cc([As](=O)(O)O)ccc1O  

Descriptors that caused failures:
MaxPartialCharge       2
MinPartialCharge       2
MaxAbsPartialCharge    2
MinAbsPartialCharge    2
BCUT2D_MWHI            4
BCUT2D_MWLOW           4
BCUT2D_CHGHI           4
BCUT2D_CHGLO           4
BCUT2D_LOGPHI          4
BCUT2D_LOGPLOW         4
BCUT2D_MRHI            4
BCUT2D_MRLOW           4
dtype: int64


NEAR-CONSTANT DESCRIPTOR FILTER

In [ ]:
# Removing low-information descriptors where more than 95% of molecules share the same value.
# Such descriptors carry negligible discriminative information for group comparison regardless of their scale or units.

desc_cols    = desc_names.copy()
low_info_desc = []

for desc in desc_cols:
    top_freq = desc_matrix[desc].value_counts(normalize=True).iloc[0]
    if top_freq > 0.95:
        low_info_desc.append(desc)

desc_cols = [d for d in desc_cols if d not in low_info_desc]

print("Low-Information Feature Filter\n")
print("Method: >95% identical values (scale-independent, outlier-resistant)\n")
print("Descriptors dropped:  ", len(low_info_desc))
print("Dropped:              ", low_info_desc)
print("Descriptors remaining:", len(desc_cols))

Low-Information Feature Filter

Method: >95% identical values (scale-independent, outlier-resistant)

Descriptors dropped:   50
Dropped:               ['NumRadicalElectrons', 'SMR_VSA2', 'SMR_VSA8', 'SlogP_VSA9', 'EState_VSA11', 'NumBridgeheadAtoms', 'NumSpiroAtoms', 'fr_Ar_COO', 'fr_C_S', 'fr_HOCCN', 'fr_Imine', 'fr_N_O', 'fr_SH', 'fr_aldehyde', 'fr_alkyl_carbamate', 'fr_amidine', 'fr_azide', 'fr_azo', 'fr_barbitur', 'fr_benzodiazepine', 'fr_diazo', 'fr_dihydropyridine', 'fr_epoxide', 'fr_furan', 'fr_guanido', 'fr_hdrzine', 'fr_hdrzone', 'fr_isocyan', 'fr_isothiocyan', 'fr_lactam', 'fr_lactone', 'fr_morpholine', 'fr_nitrile', 'fr_nitro', 'fr_nitro_arom', 'fr_nitro_arom_nonortho', 'fr_nitroso', 'fr_oxazole', 'fr_oxime', 'fr_phos_acid', 'fr_phos_ester', 'fr_priamide', 'fr_prisulfonamd', 'fr_quatN', 'fr_sulfone', 'fr_term_acetylene', 'fr_tetrazole', 'fr_thiazole', 'fr_thiocyan', 'fr_thiophene']
Descriptors remaining: 167


IMID-BIAS FILTER

In [ ]:
# Substructure-based IMID flagging via glutarimide ring match.
IMID_SMARTS = Chem.MolFromSmarts("O=C1CCCC(=O)N1")

def is_imid(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False
        return mol.HasSubstructMatch(IMID_SMARTS)
    except:
        return False

glue_mask                              = desc_matrix["Label"] == 1
desc_matrix["is_IMID"]                 = False
desc_matrix.loc[glue_mask, "is_IMID"] = desc_matrix.loc[glue_mask, "Canonical_SMILES"].apply(is_imid)

imid_count    = desc_matrix[glue_mask & (desc_matrix["is_IMID"] == True)].shape[0]
nonimid_count = desc_matrix[glue_mask & (desc_matrix["is_IMID"] == False)].shape[0]

print("IMID Identification\n")
print("IMIDs in glue set:", imid_count)
print("Non-IMIDs in glue set:", nonimid_count)

IMID Identification

IMIDs in glue set: 1077
Non-IMIDs in glue set: 546


In [ ]:
# Flagging and dropping descriptors driven by IMID-specific structure, not general glue-likeness.

# A descriptor is flagged as IMID-biased if BOTH conditions hold:
# Condition 1: |d(non-IMID vs BG)| < 0.5 → non-IMIDs look similar to background
# Condition 2: |d(IMID vs BG)|     > 0.5 → IMIDs are the outlier driving separation

def cohens_d(g1, g2):
    n1, n2 = len(g1), len(g2)
    if n1 < 2 or n2 < 2:
        return 0
    v1, v2     = np.var(g1, ddof=1), np.var(g2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0
    return (np.mean(g1) - np.mean(g2)) / pooled_std

imid_data    = desc_matrix[(desc_matrix["Label"] == 1) & (desc_matrix["is_IMID"] == True)]
nonimid_data = desc_matrix[(desc_matrix["Label"] == 1) & (desc_matrix["is_IMID"] == False)]
bg_data      = desc_matrix[desc_matrix["Label"] == 0]

imid_biased  = []
bias_results = []

for desc in desc_cols:
    d1 = cohens_d(nonimid_data[desc].values, bg_data[desc].values)  # non-IMID vs BG
    d2 = cohens_d(imid_data[desc].values,    bg_data[desc].values)  # IMID vs BG

    biased = (abs(d1) < D1_THRESHOLD) and (abs(d2) > D2_THRESHOLD)

    bias_results.append({
        "Descriptor":      desc,
        "d_nonIMID_vs_BG": round(d1, 3),
        "d_IMID_vs_BG":    round(d2, 3),
        "IMID_Biased":     biased
    })

    if biased:
        imid_biased.append(desc)

bias_df   = pd.DataFrame(bias_results)
desc_cols = [d for d in desc_cols if d not in imid_biased]

print("IMID Bias Filter\n")
print("Descriptors dropped:", len(imid_biased))
print("Descriptors remaining:", len(desc_cols))
print("\nDropped descriptors:")
print(bias_df[bias_df["IMID_Biased"]][["Descriptor", "d_nonIMID_vs_BG", "d_IMID_vs_BG"]].to_string(index=False))

IMID Bias Filter

Descriptors dropped: 33
Descriptors remaining: 134

Dropped descriptors:
                     Descriptor  d_nonIMID_vs_BG  d_IMID_vs_BG
              MaxAbsEStateIndex            0.109         0.690
                 MaxEStateIndex            0.109         0.690
               FpDensityMorgan3            0.402         0.518
                 BCUT2D_LOGPLOW            0.145        -0.839
                          Chi2n            0.474         0.643
                          Chi3n            0.319         0.588
                          Chi3v            0.367         0.500
                          Chi4n            0.182         0.510
                     PEOE_VSA13            0.353         1.645
                     PEOE_VSA14           -0.286        -0.620
                      PEOE_VSA2            0.455         2.849
                       SMR_VSA1           -0.428         0.509
                       SMR_VSA4           -0.282        -0.533
                     SlogP_

In [ ]:
# Sensitivity check for IMID bias filter at different thresholds

print("Sensitivity Check: IMID Bias Filter\n")
print(f"{'Threshold':<12} {'Descriptors Flagged':<22} {'Descriptors Remaining'}")
print("-" * 50)

for d1_thresh, d2_thresh in [(0.4, 0.4), (0.5, 0.5), (0.6, 0.6)]:
    flagged = []
    for _, row in bias_df.iterrows():
        if (abs(row["d_nonIMID_vs_BG"]) < d1_thresh) and \
           (abs(row["d_IMID_vs_BG"])    > d2_thresh):
            flagged.append(row["Descriptor"])
    print(f"{d1_thresh:<12} {len(flagged):<22} {len(bias_df) - len(flagged)}")

Sensitivity Check: IMID Bias Filter

Threshold    Descriptors Flagged    Descriptors Remaining
--------------------------------------------------
0.4          37                     130
0.5          32                     135
0.6          28                     139


SUMMARY OF RESULTS

In [ ]:
summary = pd.DataFrame({
    "Stage": [
        "Initial descriptors",
        "After near-constant descriptor filter",
        "After IMID bias filter",
        "Final descriptor count"
    ],
    "Descriptors": [
        len(desc_names),
        len(desc_names) - len(low_var_desc),
        len(desc_names) - len(low_var_desc) - len(imid_biased),
        len(desc_cols)
    ]
})
print("Summary of Results\n")
print(summary.to_string(index=False))

Summary of Results

                                Stage  Descriptors
                  Initial descriptors          217
After near-constant descriptor filter          167
               After IMID bias filter          134
               Final descriptor count          134


SAVING RESULTS TO GOOGLE DRIVE

In [ ]:
# Resetting Master_ID after rows were dropped during descriptor calculation.
final_desc_matrix = desc_matrix[["Master_ID", "Source_ID", "Canonical_SMILES", "StdInChIKey", "Label"] + desc_cols].copy()
final_desc_matrix["Master_ID"] = range(1, len(final_desc_matrix) + 1)

# Saving final descriptor matrix and IMID bias results to Google Drive.
OUT_MATRIX = DRIVE_DIR + "/descriptor_matrix.csv"
OUT_BIAS   = DRIVE_DIR + "/imid_bias_results.csv"

final_desc_matrix.to_csv(OUT_MATRIX, index=False)
bias_df.to_csv(OUT_BIAS, index=False)

print("Descriptor matrix saved to:", OUT_MATRIX)
print("IMID bias results saved to:", OUT_BIAS)
print("File exists:", os.path.exists(OUT_MATRIX))

print("\nFinal descriptor matrix shape:", final_desc_matrix.shape)
print("Final set size:", len(final_desc_matrix))
print(final_desc_matrix["Label"].value_counts())

Descriptor matrix saved to: /content/drive/MyDrive/Thesis Datasets/descriptor_matrix.csv
IMID bias results saved to: /content/drive/MyDrive/Thesis Datasets/imid_bias_results.csv
File exists: True

Final descriptor matrix shape: (3738, 139)
Final set size: 3738
Label
0    2115
1    1623
Name: count, dtype: int64
